# 最新 AI をプログラムで操ろう！ (Nano Banana 2 / API 実践)

本日は Google Colab を使用して、最新のマルチモーダル AI を「プログラムから操作する」という体験をします。
普段ブラウザで見ている AI の裏側で、どんなやり取りが行われているのかを、手を動かしながら解き明かしていきましょう。

## 1. 必要な道具（SDK）のインストール

In [ ]:
!pip install --upgrade google-genai

### 2. Google Cloud を使うための認証

今日のハンズオンでは Google Cloud を利用します。

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
print("準備が完了しました！")


## API とは何か？

本日のゴールは、Web ブラウザからではなく、皆さんが書くプログラム上から Vertex AI などの便利なサービスの機能を自由自在に呼び出して使う方法 を理解することです。

そのために欠かせないのが API (Application Programming Interface) という仕組みです。

### API の定義

API とは、あるソフトウェアの機能を、外部の別のプログラムから利用できるようにするための約束事（ルール）です。

### API の役割

1. 他のソフトウェアへの窓口: あるサービスやソフトウェアの便利な機能を他のソフトウェアに提供します。
2. リクエストとレスポンス: どんな形式で依頼し、どんな形式で結果が返るか を厳密に定めます。
3. 実装の隠蔽: 内部の実装を知らなくても、決められた通りに頼めば結果が得られます。

### Web API という仕組み

API には様々な種類がありますが、本日はインターネットを通じて機能を利用する Web API (REST API) を使用します。

Web API の最大の特徴は、HTTP という普段皆さんが Web サイトを見るときの通信の仕組みをそのまま使っていることです。

だからこそ、最新の AI を操るための第一歩として、まずは皆さんが普段使っている Web の正体 を紐解くところから始めましょう。

## Web ページの正体は テキスト の集まり

私たちが普段ブラウザ（Chromeなど）で見ている Web ページは、記号の集まったテキストデータでできています。

ブラウザがそのテキストを読み取って、人間が見やすいように色をつけたり画像を配置したりしてくれているだけなのです。

まずは、インターネットを通じてその生のテキストデータ を直接取得してみましょう。

### ブラウザを使わずに、コマンドで example.com の内容を取得してみる

[example.com](http://example.com) はドキュメントなどにサンプルとして記すための特別なサイトです。まずはこのサイトをブラウザで開きましょう。

次に、ブラウザで元のテキストを表示してみましょう。Chrome では「ページのソースを表示」したり開発者ツールを起動することで確認できます。

では、同じことをコマンドで実行してみましょう。次のコマンドを実行すると、サーバーにアクセスして Web ページのソースを取得できます。

In [ ]:
!curl -v https://example.com

実行結果は次のようになります。

```
curl -v http://example.com
* Host example.com:80 was resolved.
* IPv6: 2606:4700:10::ac42:93f3, 2606:4700:10::6814:179a
* IPv4: 104.20.23.154, 172.66.147.243
*   Trying [2606:4700:10::ac42:93f3]:80...
* Connected to example.com (2606:4700:10::ac42:93f3) port 80
> GET / HTTP/1.1
> Host: example.com
> User-Agent: curl/8.7.1
> Accept: */*
> 
* Request completely sent off
< HTTP/1.1 200 OK
< Date: Tue, 05 May 2026 05:32:30 GMT
< Content-Type: text/html
< Transfer-Encoding: chunked
< Connection: keep-alive
< Server: cloudflare
< Last-Modified: Fri, 01 May 2026 01:24:29 GMT
< Allow: GET, HEAD
< Accept-Ranges: bytes
< Age: 4015
< cf-cache-status: HIT
< CF-RAY: 9f6d5d727e18f6fe-NRT
< 
<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>
* Connection #0 to host example.com left intact
```

[cURL](https://curl.se/) (カール) は、ブラウザを使わずにコマンドラインからインターネットを通じてデータのやり取りを行うための道具です。

実行結果の中身を詳しく見てみましょう。HTTP 通信は大きく分けて「リクエスト（依頼）」と「レスポンス（返答）」の2つで構成されています。

### 1. リクエスト (Request)
> で始まる行が、皆さんのPCからサーバーへ送られた内容です。
- GET / HTTP/1.1: /（トップページ）を GET（ください）というお願いです。
- Host: example.com: お願い先のサーバーの名前です。

### 2. レスポンス (Response)
< で始まる行が、サーバーから皆さんのPCへ返ってきた内容です。
- HTTP/1.1 200 OK: 200 は成功しましたという合言葉（ステータスコード）です。
- Content-Type: text/html: 返ってきたデータの中身が HTML という形式のテキストであることを表しています。

### 3. ボディ (Body)
最後に表示されている <!doctype html>... で始まる長いテキストが、通信の中身そのものです。これをブラウザが読み取ると、私たちがよく知る Web サイトの画面になります。

このように、ブラウザを使わなくてもテキストでの会話（HTTP 通信）だけで Web サイトの情報が手に入ることがわかりました。Web API も、これと全く同じ仕組みを利用しています。


## API と会話するための共通ルール

Web API を通じて外部の機能を部品として使うためには、世界中で決まっている共通のルールに従う必要があります。それが REST API という考え方です。

REST API では、主に以下の 2 つの要素を組み合わせて通信を行います。

### 1. HTTP メソッド（動詞）
サーバーに対して何をしたいかを表す記号です。
- GET (取得): 情報をくださいと頼むときに使います。（例：Web ページを見る、猫の豆知識をもらう）
- POST (作成・処理): こちらからデータを送って処理してと頼むときに使います。（例：AI にプロンプトを送って画像を作ってもらう）

### 2. JSON (名詞・データ形式)
コンピュータ同士がデータを効率よくやり取りするための共通言語です。
先ほど見た HTML は人間が見るためのレイアウト情報などが含まれていましたが、JSON (JavaScript Object Notation) は純粋なデータだけが詰まったシンプルなテキスト形式です。

次は、実際にデータだけを返す本物の API を叩いて、HTML との違いを体験してみましょう。

### 実践 1：公開 API から情報を取得してみよう (GET)

ここでは catfact.ninja という公開 API を使ってみます。これは、実行するたびに「猫に関する豆知識」を 1 つランダムに返してくれるシンプルなサービスです。

情報を「ください」とお願いするので、HTTP メソッドは GET を使います。

In [ ]:
!curl https://catfact.ninja/fact


実行すると、以下のようなテキストが返ってきます（内容は実行するたびに変わります）。

```json
{"fact":"Statistics indicate that animal lovers in recent years have shown a preference for cats over dogs!","length":98}
```

これが JSON 形式のデータです。JSON は次のような形式をしています。

```json
{
    "key1":"value1",
    "key2":"value2",
    :
}
```

キー ("key1", "key2") や値 ("value1", "value2") はさまざまな値を自由に設定できます。また、値には文字列だけではなく、数値などのさまざまなデータを指定できます。今回は次のようになっています。

- fact: 豆知識の本文
- length: その文字の長さ

HTML とは違い、余計な装飾がなく、コンピューターが処理しやすいように「項目名」と「内容」がセットになって並んでいることがわかります。

JSON の仕様は次のサイトで確認できます。

[https://www.json.org/json-ja.html](https://www.json.org/json-ja.html)

### 実践 2：こちらからデータを送ってみよう (POST)

次は、こちらからサーバーへデータを送信する POST リクエストを体験します。これは、AI に対して「このプロンプトで画像を作って！」と依頼する際の通信と同じ仕組みです。

テスト用のサービス `httpbin.org` を使って、JSON データを送ってみましょう。


In [ ]:
# -X POST: POST メソッドを使う指定
# -H ...: 「今から JSON を送るよ」という宣言（ヘッダー）
# -d ...: 送る中身（データ）
!curl -X POST -H "Content-Type: application/json" -d '{"message": "hello IPUT!"}' https://httpbin.org/post

実行結果は次のようになります。

```json
{
  "args": {}, 
  "data": "{\"message\": \"hello IPUT!\"}", 
  "files": {}, 
  "form": {}, 
  "headers": {
    "Accept": "*/*", 
    "Content-Length": "26", 
    "Content-Type": "application/json", 
    "Host": "httpbin.org", 
    "User-Agent": "curl/8.7.1", 
    "X-Amzn-Trace-Id": "Root=1-69f987b6-63ddd29d7aa954bb692afa6b"
  }, 
  "json": {
    "message": "hello IPUT!"
  }, 
  "origin": "123.225.37.37", 
  "url": "https://httpbin.org/post"
}
```

実行結果の中に、以下のような箇所が見つかるはずです。

```json
"json": {
    "message": "hello IPUT!"
}
```

これは、サーバーが「あなたの送った JSON データ（hello IPUT!）を正しく受け取りましたよ」と教えてくれているものです。

このように、API を使うことで「情報を取得する」だけでなく「こちらからデータを送信して処理してもらう」ことが可能になります。

### Python の 辞書 と JSON

先ほどの実践で、POST リクエストを送る際に `{"message": "hello IPUT!"}` というデータを指定しました。

この書き方は、Python の **辞書型 (dict)** というデータの持ち方と同じです。

#### データの共通言語

- **JSON**: インターネット上でデータをやり取りするための「テキスト形式」の共通ルール。
- **辞書型**: Python でのデータの一種。プログラムでは文字で記述できるものの、Python では文字列とは厳密に別もの。

API を使うプログラムの裏側では、以下のような変換が常に行われています。

1. **シリアライズ (変換)**: Python のデータを JSON（ただの文字列）に変換してインターネットへ送る。
2. **デシリアライズ (復元)**: インターネットから届いた JSON（ただの文字列）を Python のデータに戻してプログラムで使う。

次は、実際に Python のコードを書いて、この「変換」と「通信」を自動で行ってみましょう。

### 実践 3：Python のコードで API を叩いてみよう (GET)

いよいよプログラム（Python）からインターネットへアクセスします。ここでは、Web 通信によく使われる `requests` というライブラリを使用します。

先ほど `curl` で行った「猫の豆知識」の取得を、Python で書いてみましょう。

In [ ]:
import requests

# 1. リクエストを送る (GET)
response = requests.get('https://catfact.ninja/fact')

# 2. 届いた JSON（テキスト）を Python のデータ（辞書）に変換する（パース）
data = response.json()

# 3. 中身を確認する
print(f"生のデータ: {response.text}")
print(f"変換後のデータ: {data}")
print(f"豆知識だけを取り出す: {data['fact']}")

#### コードのポイント

- `requests.get()`: 指定した URL に「情報をください」と通信します。
- `response.json()`: 届いた JSON 文字列を自動的に解析（パース）して、Python の辞書型にしてくれます。

これにより、インターネット上のデータが、皆さんが知っている Python のリストや辞書と同じように扱えるようになりました。

### 実践 4：Python のコードでデータを送ってみよう (POST)

次は Python のコードを使って、サーバーへデータを送信してみましょう。
先ほど `curl` コマンドで送った内容と同じものを、`requests` ライブラリを使って書いてみます。

In [ ]:
import requests

# 送りたいデータ（辞書型）
payload = {"message": "hello from Python!"}

# リクエストを送る (POST)
# 引数 json= に辞書を渡すと、自動的に JSON 文字列に変換して送ってくれます
response = requests.post('https://httpbin.org/post', json=payload)

# 結果を確認する
data = response.json()
print(f"送信したデータがサーバーで認識されました: {data}")
print(f"送信したデータを次のように送り返しています: {data['json']}")

### コードのポイント

- `json=payload`: これだけで、Python の辞書を JSON にシリアライズして送信してくれます。
- `data['json']`: サーバー（httpbin.org）が「受け取った JSON」を整理して返してくれている部分を確認しています。

これで、「自分の書いたコード」からインターネットを通じて「外部の機能」へデータを送り、結果を受け取ることができました。AI にプロンプトを送る際も、基本的にはこれと同じことが行われています。

## 公式リファレンスを見てみよう

API の正体が「テキストでの会話」であることがわかってきました。
では、いよいよ本日の主役である Google Cloud の最新 AI「Gemini」の API 仕様を見てみましょう。

### Gemini API (generateContent) の仕様

AI に対して「テキストや画像を生成して！」と頼むための正式なルールが、以下の公式ドキュメントに記されています。

[Vertex AI API リファレンス (generateContent)](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/reference/rest/v1/projects.locations.publishers.models/generateContent)

### ドキュメントを見てみよう

URL を開いて、Request body という項目を探してみてください。そこには、AI にたった一つの指示を送るために必要な、非常に複雑で階層の深い JSON の構造が書かれています。

これを毎回 requests ライブラリを使って、手書きで辞書を作って送るのは、次のような課題に直面します。

- 階層が深すぎて、どこに何を書けばいいか間違える。
- プロジェクトIDや認証トークンを毎回管理するのが面倒。

「もっと楽に、スマートに API を使いたい！」
そんな開発者の願いを叶えるための道具である SDK (Software Development Kit) を次に紹介します。

## SDK を使ってみよう

先ほど見たように、生の Web API を直接扱うのは、URL の組み立てや複雑な JSON の作成など、非常に手間がかかります。

そこで登場するのが **SDK (Software Development Kit)** です。

### SDK とは何か？

SDK とは、特定の Web API を特定のプログラミング言語から簡単に使うための「道具箱（ライブラリ）」です。

今回のワークショップでは、Web API を Python から自在に操作するための **「専用リモコン（ラッパー）」** として機能します。

### SDK を使うメリット

1. **複雑な手順の自動化**: URL の組み立て、認証ヘッダーの付与、JSON への変換（パース）などを、裏側ですべて自動でやってくれます。
2. **直感的なコード**: `requests` では何十行もかかっていた複雑な通信を、Python の関数を 1 つ呼び出すだけで実行できます。
3. **ミスの防止**: 階層の深い JSON を手書きする必要がないため、入力ミスや構造のエラーを劇的に減らすことができます。

これからは、Google 公式の SDK を導入して、先ほど見た「絶望するほど複雑な API」を、使いこなしてみましょう。

### SDK を使って AI に接続しよう（Client の作成）

いよいよ Google の最新 AI に接続するための準備をします。ここでは genai.Client を使って、自分専用の client を作成します。

#### なぜ Client を作るのか？

API を使うには、毎回「どのプロジェクトを使うのか (プロジェクトID)」「どこに通信するか (ロケーション)」「認証パスワード」などをサーバーに伝える必要があります。これを漏れなく毎回書くのは大変です。

そこで、これらの設定をすべて記憶した「あなた専属の AI 通信担当者（client）」を定められた方法で作っておきます。

これ以降は、この担当者に対して「テキストを作って」「画像を作って」と指示を出すだけで、面倒な通信の手続きをすべて裏側で自動的にやってくれるようになります。

In [ ]:
from google import genai
from google.colab import auth

# 認証の実行（まだの場合はここでポップアップが出ます）
auth.authenticate_user()

# あなたのプロジェクト ID を入力してください
PROJECT_ID = "YOUR_PROJECT_ID" # @param {type:"string"}

# SDK のクライアント（担当者）を初期化
client = genai.Client(
    vertexai=True, 
    project=PROJECT_ID, 
    location="global",
)

print(f"プロジェクト {PROJECT_ID} で AI への接続担当者（client）が準備できました！")

## Gemini (nanobanana) を用いた画像生成

ここまでの内容を踏まえて、Gemini (Nano Banana 2) を使って、実際に画像を生成してみましょう。

普段アプリから行っている画像生成を、プログラムから行う方法をマスターしましょう。

### 本日の実践フロー

画像生成を行う方法も、これまで見てきた Web API の利用方法と同じです。具体的には次のことを行います。

1. AI に「どんな画像を作りたいか」を伝える (リクエスト)。
2. AI が言葉で説明しつつ、画像を生成する (レスポンス)。
3. 届いたデータの中から、画像データを取り出して表示する。

それでは、世界最先端の AI を皆さんのプログラムに組み込んでみましょう！

### 実践：Gemini に画像を生成してもらおう

それでは、実際にプロンプトを送って画像を生成します。
このコードでは、大きく分けて「誰に・どう頼むか」と「どんな材料を渡すか」を定義しています。

#### 誰に・どう頼むか (通信先の指定)

- `client.models`:
  先ほど作った専用担当者（client）が持つ、AI モデルを扱うための専門部署です。
- `generate_content(...)`:
  その専門部署に対して「コンテンツを作って！」と命令（POSTリクエスト）を実行する関数です。このカッコ `()` の中に、以下の「材料（引数）」を渡します。

#### どんな材料を渡すか (generate_content の引数)

- 引数 `model=`: どの AI モデルを使うかを指定します。ここでは `MODEL_ID` に入れた画像生成モデルを指定しています。
- 引数 `contents=`: AI に送るメッセージ本体です。ここでは `prompt` 変数に入れた「青く光るサイバーパンクな猫」というテキストを渡しています。
- 引数 `config=`: API に送る「細かい設定書」です。今回は `GenerateContentConfig` を使って、「テキストと一緒に画像 (`Modality.IMAGE`) も出力してOK」という特別な許可を与えています。

設定をまとめたオブジェクトを作成して引数に渡すというのは、慣れるまで奇妙な感じがするかもしれません。この方法は設定項目が多すぎたり、階層構造を持っていて複雑だったり、デフォルト値が多い場合に行われます。もし、デフォルトとはちょっと違う挙動をさせたい場合には、このようなオブジェクトを用いてどのような設定ができるのか、調べてみるとよいでしょう。

それでは、コードを実行して AI に依頼を送ってみましょう。

In [ ]:
from google.genai.types import GenerateContentConfig, Modality

# 今回使うモデルの名前（Nano Banana 2）
MODEL_ID = "gemini-3.1-flash-image-preview-001"

# あなたが作りたい画像の説明を書いてください
prompt = "青く光るサイバーパンクな猫"

print("AIに依頼を送信しました。考えながら絵を描いています...（数十秒かかります）")

# 専用担当者（client）を通じて AI にリクエストを送信
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=GenerateContentConfig(
        # AIに対して「テキスト」と「画像」の両方を返してよいと許可する設定
        response_modalities=[Modality.TEXT, Modality.IMAGE],
    ),
)

print("生成が完了し、レスポンスが手元に届きました！次は中身を見てみましょう。")

### レスポンスの「解剖」

生成が完了しました！
では、AIから返ってきたデータ（`response`）の中身を覗いてみましょう。

In [ ]:
# 返ってきたデータ全体を表示してみる
print(response)


非常に複雑な構造に見えますが、実は SDK がすでに「Pythonのオブジェクト」としてサーバーからの出力を整理してくれたあとの状態です。

先ほどの Step 2.5 で `requests` を使ったときは、`response.json()` という処理（パース）を自分で書く必要がありました。しかし SDK を使えば、その処理はすでに終わっています。

必要なデータを取り出すには、辞書のように探すのではなく、ドット(`.`)で繋ぐだけで直感的にアクセスできます。

In [ ]:
# 例：AIが考えたこと（テキスト）だけを取り出す
print("--- AIのコメント ---")
for part in response.candidates[0].content.parts:
    if part.text:
        print(part.text)

### 生成された画像をファイルとして保存しよう

レスポンスの中には、テキストだけでなく「画像データ」も含まれています。
しかし、AI から届いたデータ (`inline_data.data`) は、コンピュータにとっての生のデータ、つまり 「0 と 1 の単なる数字の羅列（バイナリデータ）」 です。
このままでは、画面に「絵」として表示することも、ファイルとして保存することもできません。

そこで、2 つの特別な道具を使います。

1. BytesIO: 数字の羅列を、プログラムの中で「あたかも画像ファイル (.pngなど) がそこにあるかのように」扱うための道具です。メモリ上に「仮想のファイル」を作るイメージです。
2. PIL (Python Imaging Library): その仮想ファイルを読み込んで、私たちが目で見える「画像」に変換し、ファイルとして書き出したり画面に表示したりしてくれる道具です。


AI から届いたデータを `PIL` で画像に変換したら、それを **「ファイルとして保存」** してみましょう。
画面に表示するだけでなくファイルに残すことで、後でダウンロードしたり、他のアプリで使ったりできるようになります。

AI は一度に複数の画像を生成することもあります。
その際、ファイル名が 1 つだけだと上書きされてしまうため、Python の enumerate という機能を使って順番に番号をつけて保存しましょう。

In [ ]:
from PIL import Image
from io import BytesIO

print("--- 生成された画像の保存 ---")

# enumerate を使うと、中身（part）と一緒に「何番目か（i）」も取得できます
for i, part in enumerate(response.candidates[0].content.parts):
    if part.inline_data:
        # 1. AIから届いた数字の羅列を仮想ファイルとして扱う (BytesIO)
        # 2. それを画像として読み込む (Image.open)
        img = Image.open(BytesIO(part.inline_data.data))

        # 3. ファイル名に番号をつけて保存する（f-string を活用）
        # 例: output_image_0.png, output_image_1.png ...
        filename = f"output_image_{i}.png"
        img.save(filename)
        
        print(f"画像を {filename} として保存しました！")

        # 4. 画面にも表示して確認
        display(img)

これで、プログラムから AI に指示を出し、結果を受け取って「ファイルとして手に入れる」までの一連の流れが完成しました！

## お疲れ様でした！

本日は、ブラウザの裏側にある「Web API」という仕組みから、最新の AI モデルを Python で自在に操る「SDK」の使い方までを一気に駆け抜けました。

### 今日できるようになったこと

- ブラウザを通さず、プログラムからインターネットと会話する。
- 複雑な JSON データを Python の辞書として扱う。
- SDK（専用リモコン）を使って、最新 AI に指示を出し、画像を生成してファイルに保存する。

### なぜ「プログラム」から AI を使うのか？

冒頭のスライドで紹介した「カプコンの事例」を思い出してください。
もし、数万件のオブジェクト案を一つずつブラウザで入力して画像を作っていたら、何ヶ月もかかってしまいます。
しかし、今日皆さんが学んだように **プログラム（API）** を使えば、リストにある何万件ものデータを読み込み、自動で AI に指示を出し、自動でファイルに保存する、という作業を数時間で終わらせることができます。

これこそが、皆さんがプログラミングを学ぶ最大の武器になります。
「こんな画像が自動で欲しかった」「この作業、AI に任せられないかな？」
そんな好奇心を大切に、ぜひ最新 AI を使い倒して面白いものを作ってみてください。

皆さんの素晴らしい作品に出会えるのを楽しみにしています！